<a href="https://colab.research.google.com/github/lilianabs/aggressiveness-detection-mex-spa/blob/main/2.Baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#%pip install emoji

In [2]:
#%pip install spacy && python -m spacy download es_core_news_sm

In [9]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

import emoji
import re
import spacy

In [11]:
from google.colab import drive
drive.mount('/content/drive')

nlp = spacy.load("es_core_news_sm")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
file_path = '/content/drive/MyDrive/data_colab/MEX-A3T/train_aggressiveness.csv'
df = pd.read_csv(file_path)
df = df.drop("Id", axis=1)
display(df.head())

,Category,Text
0,0,Soy el Clint Eastwood de los Puentes de Madison en todas las putas historias de amor que me han tocado.\n
1,0,"Actualmente ya pasó de moda la pucha joto, ahora sólo quedamos los verdaderos seguidores, los que amamos este estilo de vida.\n"
2,0,"¿Es cierto esto? Y no me refiero a lo que dijo, ni al tweet, sino a👉<URL>¿Están tratando de sacarlo del programa?\n"
3,0,Vuela pega y esquiva... la neta está de la vergaaaa pero es pegajosa!!\n
4,0,Mejor puto disfraz de la noche!!!! 👊👊👊Por tercer año consecutivo morros\n


## Data cleaning

In [6]:

def remove_emojis(text):
  return emoji.replace_emoji(text, replace='')

df['Text'] = df['Text'].apply(remove_emojis)
display(df.head())

,Category,Text
0,0,Soy el Clint Eastwood de los Puentes de Madison en todas las putas historias de amor que me han tocado.\n
1,0,"Actualmente ya pasó de moda la pucha joto, ahora sólo quedamos los verdaderos seguidores, los que amamos este estilo de vida.\n"
2,0,"¿Es cierto esto? Y no me refiero a lo que dijo, ni al tweet, sino a<URL>¿Están tratando de sacarlo del programa?\n"
3,0,Vuela pega y esquiva... la neta está de la vergaaaa pero es pegajosa!!\n
4,0,Mejor puto disfraz de la noche!!!! Por tercer año consecutivo morros\n


In [13]:
def clean_text(text):
    text = text.lower()
    text = text.replace('\n', '')
    text = text.replace('@usuario', '')
    text = text.replace('<url>', '')
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

df['Text'] = df['Text'].apply(clean_text)
display(df.head())

,Category,Text
0,0,soy el clint eastwood de los puentes de madison en todas las putas historias de amor que me han tocado
1,0,actualmente ya pas de moda la pucha joto ahora slo quedamos los verdaderos seguidores los que amamos este estilo de vida
2,0,es cierto esto y no me refiero a lo que dijo ni al tweet sino aurlestn tratando de sacarlo del programa
3,0,vuela pega y esquiva la neta est de la vergaaaa pero es pegajosa
4,0,mejor puto disfraz de la noche por tercer ao consecutivo morros


In [14]:
def remove_stop_words(text):
    doc = nlp(text)
    tokens = [token.text for token in doc if not token.is_stop]
    return " ".join(token for token in tokens)

In [15]:
df['Text'] = df['Text'].apply(remove_stop_words)
display(df.head())

,Category,Text
0,0,clint eastwood puentes madison putas historias amor tocado
1,0,actualmente pas moda pucha joto slo quedamos verdaderos seguidores amamos estilo vida
2,0,refiero tweet aurlestn tratando sacarlo programa
3,0,vuela pega esquiva neta est vergaaaa pegajosa
4,0,puto disfraz noche tercer ao consecutivo morros


In [16]:
from sklearn.model_selection import train_test_split

X = df['Text']
y = df['Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Tf-idf vectorization

### Subtask:
Convert the text data into numerical features using TF-IDF.


**Reasoning**:
Import the TfidfVectorizer and transform the training and testing data using TF-IDF.



In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Shape of X_train_tfidf:", X_train_tfidf.shape)
print("Shape of X_test_tfidf:", X_test_tfidf.shape)

Shape of X_train_tfidf: (5865, 13054)
Shape of X_test_tfidf: (1467, 13054)


In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

LogisticRegression()

## Evaluate model

### Subtask:
Evaluate the performance of the trained model.


**Reasoning**:
Import necessary metrics and calculate the evaluation metrics for the trained model.



In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")

Accuracy: 0.8029993183367417
Precision: 0.8059701492537313
Recall: 0.4768211920529801
F1-score: 0.5991678224687933


## Summary:

### Data Analysis Key Findings

*   The dataset was split into training (5865 samples) and testing (1467 samples) sets, with 80% for training and 20% for testing.
*   TF-IDF vectorization transformed the text data into numerical features, resulting in 13054 unique terms (features) in the training data.
*   A logistic regression model was trained on the TF-IDF transformed training data.
*   The model's performance on the test set was evaluated, yielding an accuracy of 0.8030, precision of 0.8060, recall of 0.4768, and an F1-score of 0.5992.

### Insights or Next Steps

*   The relatively low recall (0.4768) suggests the model struggles to identify all positive cases. Investigating techniques to improve recall, such as adjusting class weights or exploring different models, could be beneficial.
*   Considering the F1-score (0.5992), which balances precision and recall, there is room for improvement in the model's overall performance. Further hyperparameter tuning of the logistic regression model or experimenting with other text classification algorithms might lead to better results.
